<a href="https://colab.research.google.com/github/UnfoldDataScience/Agentic_Ai_For_Beginner/blob/main/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

In [1]:
!python --version

Python 3.13.15


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   TAVILY_API_KEY

In [1]:
!pip install -q -U \
  "langchain==0.3.14" \
  "langchain-openai==0.2.14" \
  "langchain-community==0.3.14" \
  "langchain-core==0.3.63" \
  "openai==1.59.6" \
  "python-dotenv==1.0.1" \
  "requests==2.32.4" \
  "beautifulsoup4>=4.13.0" \
  "wikipedia==1.4.0" \
  "tavily-python==0.5.0" \
  "ipykernel==6.17.1"

In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [3]:
import requests
from bs4 import BeautifulSoup

In [4]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

Hello, David! How can I help you today?


In [7]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

The capital of the United States is Washington, D.C.


In [8]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

Deep learning is a subset of machine learning that focuses on neural networks with many layers, enabling the modeling of complex structures and patterns in data. Given the rapid advancements in this field, here is a list of essential topics to learn in deep learning, along with their importance:

### 1. **Neural Networks Fundamentals**
   - **Importance**: Understanding the architecture of neural networks is crucial as it forms the foundation for all deep learning models. This includes concepts like neurons, activation functions, and layers.

### 2. **Convolutional Neural Networks (CNNs)**
   - **Importance**: CNNs are essential for image processing and computer vision tasks. They have revolutionized fields such as image recognition, object detection, and segmentation, especially with advancements like transfer learning and architectures like ResNet and EfficientNet.

### 3. **Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM)**
   - **Importance**: RNNs and their varia

In [9]:
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a simple custom tool
def my_tool_function(query: str) -> str:
    return f"Tool response: {query}"

# Create tool from function
my_tool = Tool.from_function(
    func=my_tool_function,
    name="simple_tool",
    description="A simple tool"
)

# Tavily Search Tool
tavily_search = TavilySearchResults(max_results=2)

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Tools list
tools = [tavily_search, my_tool]

# Create agent
agent = initialize_agent(
    tools,
    llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Run agent
response = agent.run("What's the weather like today in London?")

print(response)



> Entering new AgentExecutor chain...
I need to find the current weather conditions in London. I'll use the search tool to look for the latest weather update.  
Action: tavily_search_results_json  
Action Input: "current weather London"  
Observation: [{'url': 'https://timesofindia.indiatimes.com/world/uk/london-weather-forecast-sunny-intervals-and-mild-temperatures-today-august-24-2026/articleshow/133451758.cms', 'content': 'London residents can expect sunny intervals and mild temperatures on August 24, 2026, with a maximum temperature of 22°C and a minimum of 16°C, making it a good day for outdoor activities. However, a slight chance of showers is anticipated later in the week, suggesting the need for an umbrella.Today, August 24, 2026, London will experience sunny intervals. The maximum temperature is forecast to reach 22°C, while the minimum temperature will be around 16°C. The average temperature for the day is expected to be 19.0°C. Humidity levels will be around 74%, and the c

In [10]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [11]:
tools = [tavily_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [12]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])



> Entering new AgentExecutor chain...
I need to gather information about the inventor of the World Wide Web and its impact. I'll start by searching for current and comprehensive information on this topic. 

Action: tavily_search_results_json  
Action Input: "Who invented the World Wide Web and its impact"  

Observation: [{'url': 'https://webfoundation.org/impact', 'content': 'Spread the wordRSSfacebooktwitterinstagramlinkedin\n\n# World Wide Web Foundation\n\n# World Wide Web Foundation\n\nDonate\n\n### Our Impact\n\n Our Work\n Our Impact\n\n## #ForEveryone\n\nSir Tim Berners-Lee changed the world: he invented the World Wide Web. He then gave the web to all of us for free – a move that sparked a global wave of creativity, collaboration and innovation never seen before.  The web has changed the world, but that free and open web is today under threat.'}, {'url': 'https://en.wikipedia.org/wiki/Tim_Berners-Lee', 'content': 'Sir Timothy John Berners-Lee (born 8 June 1955), also known as